# VieNeu TTS cho Sách nói GameVui

Chạy lần lượt 3 ô bên dưới (Runtime ▸ Run all). Cuối cùng ô 3 in ra một **địa chỉ https://….trycloudflare.com**. Dán địa chỉ đó vào Sách nói ▸ nút **⚙ VieNeu**.

- Dùng CPU là đủ (không cần đổi sang GPU).
- Giữ tab Colab mở khi nghe; Colab tự ngắt sau một thời gian và mỗi lần chạy lại địa chỉ sẽ khác.
- Chưa được thử với VieNeu thật: nếu ô 2 hoặc 3 báo lỗi, gửi lỗi cho người phát triển.

In [ ]:
# Ô 1: cài thư viện (lần đầu mất vài phút)
!pip -q install vieneu fastapi uvicorn
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
print('Xong')

In [ ]:
# Ô 2: ghi file máy chủ
%%writefile server.py
"""Máy chủ nhỏ bọc VieNeu-TTS cho trang Sách nói của GameVui.
Chạy:  pip install vieneu fastapi uvicorn
       python server.py            (mặc định cổng 8000)
Trang gọi:  POST /tts  {"text": "...", "voice": "tên giọng (tuỳ chọn)"}  ->  audio/wav
"""
import os, tempfile
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response
from pydantic import BaseModel
from vieneu import Vieneu

# Chỉ cho phép trang web của bạn gọi (đổi nếu cần); thêm http://localhost để thử cục bộ.
ORIGINS = os.environ.get("ALLOW_ORIGINS", "https://huyenit2016.github.io,http://localhost:8080,http://127.0.0.1:8080").split(",")

app = FastAPI(title="VieNeu cho GameVui")
app.add_middleware(CORSMiddleware, allow_origins=ORIGINS, allow_methods=["*"], allow_headers=["*"])
tts = Vieneu()  # tải mô hình ONNX chạy CPU lần đầu


class Req(BaseModel):
    text: str
    voice: str | None = None


@app.get("/health")
def health():
    return {"ok": True}


@app.post("/tts")
def synth(r: Req):
    text = r.text.strip()
    if not text or len(text) > 600:
        raise HTTPException(400, "Văn bản rỗng hoặc quá dài")
    kw = {"text": text}
    if r.voice:
        kw["voice"] = tts.get_preset_voice(r.voice)
    audio = tts.infer(**kw)
    fd, path = tempfile.mkstemp(suffix=".wav"); os.close(fd)
    try:
        tts.save(audio, path)
        return Response(open(path, "rb").read(), media_type="audio/wav")
    finally:
        os.remove(path)


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="127.0.0.1", port=int(os.environ.get("PORT", 8000)))


In [ ]:
# Ô 3: chạy máy chủ + mở đường hầm https
import subprocess, time, re, os, urllib.request
os.environ['ALLOW_ORIGINS'] = 'https://huyenit2016.github.io'
srv = subprocess.Popen(['python', 'server.py'], stdout=open('server.log', 'w'), stderr=subprocess.STDOUT)
for i in range(120):  # chờ tải mô hình tối đa ~4 phút
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/health', timeout=2); break
    except Exception:
        if srv.poll() is not None: print(open('server.log').read()[-3000:]); raise SystemExit('Máy chủ lỗi, xem log ở trên')
        time.sleep(2)
else:
    print(open('server.log').read()[-3000:]); raise SystemExit('Hết thời gian chờ máy chủ')
tun = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'], stdout=open('tunnel.log', 'w'), stderr=subprocess.STDOUT)
url = None
for i in range(40):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read())
    if m: url = m.group(0); break
    time.sleep(1)
print('\n=== Dán địa chỉ này vào Sách nói ▸ ⚙ VieNeu ===\n' + (url or 'Không lấy được địa chỉ, xem tunnel.log'))
print('Giữ ô này chạy. Dừng: Runtime ▸ Interrupt.')
try:
    while True: time.sleep(60)
except KeyboardInterrupt:
    srv.terminate(); tun.terminate()